# Search validation targets - March 29, 2016
Search the most recent DDA dataset for peptides that are potential targets for PRM validation
## Initializations

In [1]:
import os
import pandas as pd

tps_dir = os.path.join('..','..','synthesis072614')
targs_file = os.path.join(tps_dir, 'druggable_kinase_targets_032816.txt')
targs_out_file = os.path.join(tps_dir, 'druggable_kinase_targets_dda_match_032816.txt')

dda_file = 'file://localhost/Users/agitter/Desktop/postdocToSave/egfr/phosphorylation-data-sets' \
    '/McKenna2014/validation/FLPIN_EGFR_DDA_pilot_corrected_032616.xlsx'
dda_out_base = os.path.join('..','..','phosphorylation-data-sets','McKenna2014','validation')
dda_out0_file = os.path.join(dda_out_base, 'FLPIN_EGFR_DDA_pilot_corrected_0min_032616.txt')
dda_out2_file = os.path.join(dda_out_base, 'FLPIN_EGFR_DDA_pilot_corrected_2min_032616.txt')
dda_out15_file = os.path.join(dda_out_base, 'FLPIN_EGFR_DDA_pilot_corrected_15min_032616.txt')

## Load the target annotations
Manually extracted from Cytoscape file Refiner_EGFR_081115.cys.  A single protein can have multiple annotations because they are specific for a single druggable kinase.

Category 0
- No need to validate, known EGFR pathway member

Category 1
- Substrates that are directly connected to the inhibited kinase which a significantly activated peptide in our original time series data (or close to the significance threshold).

Category 2
- Proteins that are two interactions from the inhibited kinase that are significantly activated *or inhibited* in the original data.

Category 2b
- Meet the category 2 criteria, but manually determined to be less promising.  For example, the first-kinase substrate interaction is predicted to be inhibiting.

Category 3 – not worth studying further
- Proteins that are directly connected to the inhibited kinase but have either significant inhibition, insignificant temporal changes, or no observed phosphorylation in the original data.

Category 4 – not worth studying further
- Proteins that are two interactions from the inhibited kinase and do not significantly change in the original data.

In [2]:
# Sets of gene ids and protein ids for targets that are category 1 or category 2
query_genes = set()
query_prots = set()

counter = 0
with open(targs_file) as targ_f:
    #Inhibitor \t Inhibited_kinase \t Target_protein \t Target_gene \t Category
    targ_f.readline()
    for line in targ_f:
        counter += 1
        parts = line.strip().split('\t')
        
        # Protein only needs to be category 1 or 2 for one of the inhibited targets
        if parts[4] == '1' or parts[4] == '2':
            query_genes.add(parts[3])
            query_prots.add(parts[2])

assert len(query_genes) == len(query_prots), 'Different number of query genes and proteins'
print 'Loaded %d target lines and %d unique protein targets' % (counter, len(query_prots))

Loaded 169 target lines and 28 unique protein targets


## Load the DDA data
** Some columns in this file have corrupted data that have been converted to dates **

Load three sheets for different time points:
- 0min_EGF
- 2min_EGF
- 15min_EGF

In [3]:
min0_df = pd.read_excel(dda_file, sheetname='0min_EGF', convert_float=False)
min2_df = pd.read_excel(dda_file, sheetname='2min_EGF', convert_float=False)
min15_df = pd.read_excel(dda_file, sheetname='15min_EGF', convert_float=False)

## Helper function to scan peptides
Check whether a row matches a target protein or target gene.  Matching Uniprot id in the group is a loose criterion and may give a match that is likley to be another related protein.  Needs to be manually verified.  Relies on the global variable all_matches.

In [4]:
def id_match(row):
    gene = str(row['Gene Symbol']).upper()
    if gene in query_genes:
        all_matches.add(gene)
        return True
    
    # Check all protein ids in the row
    prots = row['protein'].upper().split(',')
    for prot in prots:
        prot_ids = prot.split('|')
        if prot_ids[2] in query_prots:
            all_matches.add(prot_ids[2])
            return True
    
    return False

## Filter the peptides to retain those that are query proteins
Write the matching peptides to text.  Also store the matching query genes or proteins.  IRS4_HUMAN matches one of the corrupted gene symbols, '4.0'.  Primary Uniprot id is H3BTC1_HUMAN.

In [5]:
# Store all genes (ids from the DDA file) that matched a query gene or protein
all_matches = set()

min0_match_df = min0_df[min0_df.apply(id_match, axis=1)]
min2_match_df = min2_df[min2_df.apply(id_match, axis=1)]
min15_match_df = min15_df[min15_df.apply(id_match, axis=1)]

min0_match_df.to_csv(dda_out0_file, sep='\t', index=False)
min2_match_df.to_csv(dda_out2_file, sep='\t', index=False)
min15_match_df.to_csv(dda_out15_file, sep='\t', index=False)

print '%d matching 0 min peptides'% len(min0_match_df)
print '%d matching 2 min peptides' % len(min2_match_df)
print '%d matching 15 min peptides' % len(min15_match_df)

307 matching 0 min peptides
324 matching 2 min peptides
364 matching 15 min peptides


## Write the kinase targets that match at least one DDA peptide
IRS4 appears twice

In [6]:
print '%d targets match a DDA peptide:' % len(all_matches)
for match in sorted(all_matches):
    print match

23 targets match a DDA peptide:
ACIN1
ACTB
ARHGEF11
BCLAF1
CBFB
CCDC50
DOCK4
EIF4B
ELF2
HTATSF1
IRS4
IRS4_HUMAN
IWS1
MKL1
PGRMC1
PKP4
RNF213
RSL1D1
STMN1
SUB1
TBA1A_HUMAN
WIPF2
ZYX


## Write a new version of the targets file that adds a column specifying whether the target matches a DDA peptide

In [7]:
with open(targs_file) as targ_f:
    with open(targs_out_file, 'w') as targ_out_f:
        #Inhibitor \t Inhibited_kinase \t Target_protein \t Target_gene \t Category
        header = targ_f.readline()
        targ_out_f.write('%s\tDDA_peptide_match\n' % (header.strip()))
        for line in targ_f:
            parts = line.strip().split('\t')

            # See if the gene of protein in this line matches a DDA peptide or if it was not checked due to the category
            if parts[4] == '1' or parts[4] == '2':
                if parts[2] in all_matches or parts[3] in all_matches:
                    dda_match = 'Yes'
                else:
                    dda_match = 'No'
            else:
                dda_match = 'Not checked'
            
            # Write the line with the new column
            targ_out_f.write('%s\t%s\n' % (line.strip(), dda_match))